# DBSCAN baseline

A booking is a core point if at least `min_samples` bookings lie within distance `eps` of it. Clusters are chains of core points and their neighbours, and everything else is noise (label −1)

## 1 Setup and data

In [20]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os
import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = 42
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)
pd.set_option("display.max_columns", None)

In [21]:
data = pd.read_csv("../data/H2_Cleaned_clustering.csv")
MODEL_COLS = list(data.columns)
X = data.to_numpy()
info = pd.read_csv("../data/H2_notdropped.csv")   # to describe clusters only
print("bookings:", len(X), " model features:", len(MODEL_COLS))

bookings: 48990  model features: 12


Noise bookings (label −1) are left out of the scores

In [22]:
rng = np.random.default_rng(RANDOM_STATE)
S = np.sort(rng.choice(len(X), size=10_000, replace=False))   # the fixed silhouette sample


def evaluate(labels):
    clustered = labels >= 0                                    # noise (-1) is left out of the scores
    sizes = np.bincount(labels[clustered]) if clustered.any() else np.array([0])
    scores = {"clusters": labels.max() + 1,
              "noise %": (~clustered).mean() * 100,
              "smallest cluster %": sizes.min() / len(labels) * 100}
    if labels.max() < 1:                                       # the scores need at least 2 clusters
        return scores
    s = S[clustered[S]]
    scores["silhouette"] = silhouette_score(X[s], labels[s])
    scores["davies_bouldin"] = davies_bouldin_score(X[clustered], labels[clustered])
    scores["calinski_harabasz"] = calinski_harabasz_score(X[clustered], labels[clustered])
    return scores

## 2 Input: first 5 principal components

In the 12 model columns, two bookings from different sales channels are always at least 1 apart. That is more than DBSCAN's default eps of 0.5, so a cluster can never mix channels.

Segment purity isthe share of clustered bookings that are in their cluster's most common channel. 1.0 means the clusters are just the channels. 

DBSCAN with default settings is tried on the 12 columns and on the first 8, 5 and 3 principal components

In [23]:
def segment_purity(labels):
    clustered = labels >= 0
    table = pd.crosstab(labels[clustered], info["Segment"][clustered])
    return table.max(axis=1).sum() / clustered.sum()


print("share of the largest channel:", round(info["Segment"].value_counts(normalize=True).iloc[0], 3))

variance = np.cumsum(PCA().fit(X).explained_variance_ratio_) * 100
rows = []
for n in [12, 8, 5, 3]:
    Z = X if n == 12 else PCA(n_components=n, random_state=RANDOM_STATE).fit_transform(X)
    labels = DBSCAN().fit(Z).labels_
    rows.append({"input": "12 model columns" if n == 12 else f"first {n} components",
                 "variance kept %": variance[n - 1], "clusters": labels.max() + 1,
                 "noise %": (labels < 0).mean() * 100, "segment purity": segment_purity(labels)})
pd.DataFrame(rows).set_index("input").round(3)

share of the largest channel: 0.673


,variance kept %,clusters,noise %,segment purity
input,,,,
12 model columns,100.000,268,6.926,1.000
first 8 components,99.305,249,6.661,0.981
first 5 components,95.315,87,2.378,0.680
first 3 components,75.052,3,0.139,0.673



DBSCAN will run on the first 5 principal components because it has the most variance kept with the best segment purity (3 components loose too much data)

In [ ]:
density_pca = PCA(n_components=5, random_state=RANDOM_STATE).fit(X)
Xd = density_pca.transform(X)

DBSCAN input: (48990, 5)


## 3 Baseline model

defaults: `eps=0.5`, `min_samples=5`, run on the 5 component input

In [26]:
baseline = DBSCAN().fit(Xd)
labels = baseline.labels_
pd.DataFrame([evaluate(labels)], index=["baseline"]).round(3)

,clusters,noise %,smallest cluster %,silhouette,davies_bouldin,calinski_harabasz
baseline,87,2.378,0.006,0.06,2.116,1129.091


In [27]:
sizes = pd.Series(labels[labels >= 0]).value_counts()
pd.DataFrame({"clusters": [len(sizes)],
              "with < 50 bookings": [(sizes < 50).sum()],
              "with >= 1% of bookings": [(sizes >= 0.01 * len(X)).sum()],
              "largest cluster %": [sizes.max() / len(X) * 100]}, index=["baseline"]).round(1)

,clusters,with < 50 bookings,with >= 1% of bookings,largest cluster %
baseline,87,35,23,10.9


## 4 Save the input and the baseline model

In [ ]:
joblib.dump(density_pca, f"{MODEL_DIR}/density_pca.joblib")       
joblib.dump(baseline, f"{MODEL_DIR}/dbscan_baseline.joblib")      

['model/dbscan_baseline.joblib']